In [1]:
import os
import pandas as pd

from utrfx.gtf_io import GTFio
from utrfx.genome import GRCh38

five_utr_mutations_fpath = os.path.join("..", "..", "..", "Downloads", "Five_prime_mutations.xlsx")
five_utr_mutations_df = pd.read_excel(five_utr_mutations_fpath, header=0)
gtf_fpath = os.path.join("..", "..", "..", "Downloads", "gencode.v47.basic.annotation.gtf")

gtf_file = GTFio(fpath=gtf_fpath)
transcripts = gtf_file.extract_five_utrs(genome_build=GRCh38)

In [2]:
print(five_utr_mutations_df.head())

    Chr  Position Ref Alt    Gene
0  chr1   9943502   A   T  NMNAT1
1  chr1   9943503   C   T  NMNAT1
2  chr1  21509427   C   T    ALPL
3  chr1  55039507   C   A   PCSK9
4  chr1  91022191   G   T  ZNF644


In [5]:
import csv
import subprocess
import tempfile

from utrfx.model import TxperGene
from utrfx.genome import GRCh38, VariantCoordinates, GenomeBuild
from utrfx.util import fetch_cdna_from_ensembl, get_five_prime_sequence
from utrfx.variant_util import AltAlleleSeq
from ViennaRNA import RNA

json_path = os.path.join("tests", "data", "Ensembl_transcript_per_gene_dictionary.json")


output_fpath = os.path.join("tests", "data", "Results_rna_folding.tsv")

results = []

def generate_probs(seq):
    lbox = []
    ubox = []

    with tempfile.TemporaryDirectory() as tmpdir:
        cwd = os.getcwd()
        os.chdir(tmpdir)

        try:
            process = subprocess.run(
                ['RNAfold', '--partfunc', '--noPS'],
                input=seq,
                text=True,
                capture_output=True,
                check=True
            )

            dotplot_file = os.path.join(tmpdir, "dot.ps")
            if not os.path.exists(dotplot_file):
                print("No se generó el archivo dot.ps.")
                return lbox, ubox

            with open(dotplot_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 4:
                        try:
                            pos1 = int(parts[0])
                            pos2 = int(parts[1])
                            prob = float(parts[2])
                            type = parts[3]

                            entry = {"pos1": pos1, "pos2": pos2, "score": prob}

                            if type == "lbox":
                                lbox.append(entry)
                            elif type == "ubox":
                                ubox.append(entry)
                        except ValueError:
                            continue 
        finally:
            os.chdir(cwd)

    return lbox, ubox

def compare_probs(lbox1, ubox1, lbox2, ubox2):
    sum_probs_first_Seq = sum([x["score"] for x in lbox1 + ubox1])
    suma_probs_second_seq = sum([x["score"] for x in lbox2 + ubox2])
    probs_diff = sum_probs_first_Seq - suma_probs_second_seq

    return probs_diff

header = ["tx_id", "MFE_diff", "Ensemble_diversity_diff", "MFE_frequency_diff", "Hamming_distance", "BP_distance", "Total_prob_diff"]

with open(output_fpath, "w", newline='') as file:
    writer = csv.writer(file, delimiter='\t')
    writer.writerow(header)

gene = None
tx_id = None
tx_id_clean = None
five_utr_cdna_sequence = None
contig = None
current_tx = None
for _, row in five_utr_mutations_df.iterrows():
    if gene is None or row["Gene"] != gene:
        gene = row["Gene"]
        tx_id = TxperGene(fpath=json_path).get_transcript_id(gene_symbol=row["Gene"])
        contig = GenomeBuild.contig_by_name(GRCh38, name= str(row["Chr"]))

        tx_id_clean = tx_id.split(".")[0]
        for tx in transcripts:
            if tx.tx_id == tx_id:
                tx_found = tx
                break
        
        assert tx_found is not None, "Transcript not found"

        tx_cdna_sequence = fetch_cdna_from_ensembl(transcript_id=tx_id_clean)
        five_utr_cdna_sequence = get_five_prime_sequence(cdna_sequence=tx_cdna_sequence, five_utrs=tx_found.five_utr)

        fc = RNA.fold_compound(five_utr_cdna_sequence)
        structure, mfe = fc.mfe()
        fc.pf()
        canonical_mfe = mfe
        canonical_structure = structure
        canonical_diversity = fc.mean_bp_distance()
        canonical_mfe_frequency = fc.pr_structure(canonical_structure)
        canonical_lbox, canonical_ubox = generate_probs(five_utr_cdna_sequence)

    variant = VariantCoordinates.from_vcf_literal(contig, row["Position"], row["Ref"], row["Alt"])
    variant_tx_id = tx_id_clean + "-" + str(row["Chr"]) + "-" + str(row["Position"]) + "-" + row["Ref"] + "-" + row["Alt"]
    variant_results = []
    variant_results.append(variant_tx_id)

    variant_alt_class = AltAlleleSeq(variant, five_utr_cdna_sequence, tx_found.five_utr)
    variant_in_cdna = variant_alt_class.check_variant_in_cdna()

    if variant_in_cdna == "Variant not in the 5'UTR of the given transcript":
        variant_results.append("Variant not in the 5'UTR of the given transcript")
    elif variant_in_cdna == "Reference alleles do not match":
        variant_results.append("Reference alleles do not match")
    else:
        variant_five_utr_sequence = variant_alt_class.prepare_alt_seq()
        fc_variant = RNA.fold_compound(variant_five_utr_sequence)
        variant_structure, variant_mfe = fc_variant.mfe()
        fc_variant.pf()
        variant_ensemble_diversity = fc_variant.mean_bp_distance()
        variant_mfe_freq = fc_variant.pr_structure(variant_structure)
        variant_lbox, variant_ubox = generate_probs(variant_five_utr_sequence)

        variant_results.append(canonical_mfe - variant_mfe)
        variant_results.append(canonical_diversity - variant_ensemble_diversity)
        variant_results.append(canonical_mfe_frequency - variant_mfe_freq)
        variant_results.append(RNA.hamming_distance(canonical_structure, variant_structure))
        variant_results.append(RNA.bp_distance(canonical_structure, variant_structure))
        variant_results.append(compare_probs(canonical_lbox, canonical_ubox, variant_lbox, variant_ubox))

    with open(output_fpath, "a", newline='') as file:
        writer = csv.writer(file, delimiter='\t')
        writer.writerow(variant_results)

FileNotFoundError: [Errno 2] No such file or directory: 'RNAfold'